# Lab | Agent & Vector store

**Change the state union dataset and replicate this lab by updating the prompts accordingly.**

One such dataset is the [sonnets.txt](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) dataset or any other data of your choice from the same git.

# Combine agents and vector stores

This notebook covers how to combine agents and vector stores. The use case for this is that you've ingested your data into a vector store and want to interact with it in an agentic manner.

The recommended method for doing so is to create a `RetrievalQA` and then use that as a tool in the overall agent. Let's take a look at doing this below. You can do this with multiple different vector DBs, and use the agent as a way to route between them. There are two different ways of doing this - you can either let the agent use the vector stores as normal tools, or you can set `return_direct=True` to really just use the agent as a router.

## Create the vector store

In [2]:
!pip install chromadb langchain langchain_community langchain_openai langchain_text_splitters langchain-classic


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [3]:
from langchain_classic.chains import RetrievalQA  # Legacy chains
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader


from langchain_classic.agents import AgentType, initialize_agent
from langchain_openai import ChatOpenAI

In [7]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [ ]:
# If you're using colab, run this
#os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY"

In [8]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


In [9]:
from pathlib import Path

relevant_parts = []
for p in Path(".").absolute().parts:
    relevant_parts.append(p)
    if relevant_parts[-3:] == ["langchain", "docs", "modules"]:
        break
doc_path = str(Path(*relevant_parts) / "state_of_the_union.txt")

In [11]:
loader = TextLoader("/Users/domiendarmont/Desktop/Ironhack/lab_week7_lang/lab-agent-vector-store/state_of_the_union.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")

In [12]:
state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [13]:
import os
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"

In [14]:
from langchain_community.document_loaders import WebBaseLoader

In [15]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [16]:
%pip install beautifulsoup4 lxml html5lib


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
# Put near where bs4 is imported
try:
    from bs4 import BeautifulSoup
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "bs4 (beautifulsoup4) is required. Install with `pip install beautifulsoup4`"
    ) from e

In [21]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

Created a chunk of size 2122, which is longer than the specified 1000
Created a chunk of size 3187, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 1256, which is longer than the specified 1000
Created a chunk of size 2321, which is longer than the specified 1000


## Create the Agent

In [22]:
# Import things that are needed generically
# Legacy agent imports
from langchain_classic.agents import AgentType, initialize_agent
from langchain_classic.tools import Tool  # or langchain.tools.Tool if using v1 Tool
from langchain_openai import OpenAI

In [23]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [24]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

/var/folders/9n/zs24nr4n1jq9k_zjkcrvfj1h0000gn/T/ipykernel_13602/1834837320.py:3: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = initialize_agent(


In [25]:
agent.run("What year did the most recent state of the union address take place?")

/var/folders/9n/zs24nr4n1jq9k_zjkcrvfj1h0000gn/T/ipykernel_13602/2106079623.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  agent.run("What year did the most recent state of the union address take place?")




> Entering new AgentExecutor chain...
I need to find out the year of the most recent State of the Union address. 
Action: State of Union QA System
Action Input: "What year did the most recent State of the Union address take place?"
Observation: I don't know.
Thought:I need to try a different approach to find the year of the most recent State of the Union address. 
Action: State of Union QA System
Action Input: "When was the most recent State of the Union address delivered?"
Observation: I don't know.
Thought:It seems that I am unable to retrieve the information about the most recent State of the Union address. I will need to conclude my search for this specific detail. 

Final Answer: I am unable to determine the year of the most recent State of the Union address.

> Finished chain.


'I am unable to determine the year of the most recent State of the Union address.'

In [26]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
I need to gather information about the advantages of using ruff compared to flake8, which is a popular Python linter. This will help me provide a comprehensive answer to the question.  
Action: Ruff QA System  
Action Input: "What are the advantages of using ruff over flake8?"  
Observation: The advantages of using Ruff over Flake8 include:

1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, which is written in Python.

2. **Comprehensive Rule Set**: Ruff implements over 800 rules, while Flake8 has a smaller set. This includes many popular Flake8 plugins re-implemented in Ruff.

3. **Automatic Fixing**: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations, which can save time during development.

4. **No Custom Rules**: Ruff does not support custom lint rules, but this means that all rules are implemented natively, reducing potential conflicts and simplifying configurati

{'input': 'Why use ruff over flake8?',
 'output': 'Ruff offers several advantages over Flake8, including better performance due to its Rust implementation, a more comprehensive rule set with over 800 rules, the ability to automatically fix lint violations, and easier configuration without the need for custom rules. Additionally, Ruff can be installed easily via PyPI without requiring a separate Rust installation, and its linter and formatter can be used independently. Overall, Ruff provides a more modern and efficient approach to linting Python code.'}

## Use the Agent solely as a router

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [27]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [28]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [29]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)



> Entering new AgentExecutor chain...
I need to find out what President Biden said about Ketanji Brown Jackson in the most recent State of the Union address. 
Action: State of Union QA System
Action Input: What did Biden say about Ketanji Brown Jackson in the State of the Union address?
Observation: In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the speech. He described her as one of the nation’s top legal minds who will continue Justice Stephen Breyer’s legacy of excellence.


> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': 'In the State of the Union address, Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the speech. He described her as one of the nation’s top legal minds who will continue Justice Stephen Breyer’s legacy of excellence.'}

In [30]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
I need to gather information about the advantages of using ruff compared to flake8, which is a popular Python linter. This will help me provide a comprehensive answer to the question.  
Action: Ruff QA System  
Action Input: "What are the advantages of using ruff over flake8?"  
Observation: The advantages of using Ruff over Flake8 include:

1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, which is implemented in Python.

2. **More Rules**: Ruff implements over 800 rules, while Flake8 has a smaller set of rules. This means Ruff can catch a wider variety of linting issues.

3. **Automatic Fixes**: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations, which can save time during development.

4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, which can simplify the setup and reduce conflicts.

5. **Simpli

{'input': 'Why use ruff over flake8?',
 'output': 'The advantages of using Ruff over Flake8 include:\n\n1. **Performance**: Ruff is implemented in Rust, which generally allows it to run faster than Flake8, which is implemented in Python.\n\n2. **More Rules**: Ruff implements over 800 rules, while Flake8 has a smaller set of rules. This means Ruff can catch a wider variety of linting issues.\n\n3. **Automatic Fixes**: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations, which can save time during development.\n\n4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins natively, which can simplify the setup and reduce conflicts.\n\n5. **Simplified Rule Selection**: Ruff uses different rule codes and prefixes to minimize conflicts across plugins, allowing users to toggle rules on or off more easily.\n\n6. **Independence from Plugins**: Ruff implements all rules natively and does not rely on third-party plugins,

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [31]:
tools = [
    Tool(
        name="State of Union QA System",
        func=state_of_union.run,
        description="useful for when you need to answer questions about the most recent state of the union address. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [32]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [33]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
)



> Entering new AgentExecutor chain...
I need to find out what tool Ruff uses to run over Jupyter Notebooks and whether the president mentioned that tool in the State of the Union address. I will first check the Ruff QA System for information about the tool used by Ruff.

Action: Ruff QA System  
Action Input: What tool does Ruff use to run over Jupyter Notebooks?  
Observation: Ruff integrates with nbQA, a tool for running linters and code formatters over Jupyter Notebooks.
Thought:Now that I know Ruff uses nbQA to run over Jupyter Notebooks, I need to check if the president mentioned nbQA in the State of the Union address.

Action: State of Union QA System  
Action Input: Did the president mention nbQA in the State of the Union address?  
Observation: I don't know.
Thought:I need to find out if the president mentioned nbQA in the State of the Union address, but I currently don't have that information. I will need to check again or look for more context. 

Action: State of Union QA S

{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?',
 'output': 'Ruff uses nbQA to run over Jupyter Notebooks, but the president did not mention nbQA or any specific tools related to coding or programming in the State of the Union address.'}